In [19]:
import pandas as pd 
import exifread
import uuid
import requests_cache
import openmeteo_requests
import os
import imgkit

from importlib import reload
import python.html_to_png as html_to_png
reload(html_to_png)
from python.html_to_png import convert_html_to_png

import inspect
assert not inspect.iscoroutinefunction(convert_html_to_png)

from playwright.sync_api import sync_playwright
from pathlib import Path 
from tqdm.auto import tqdm 
from retry_requests import retry
from datetime import datetime

In [20]:
# Source dirctory for original images / photos 
images_dir = Path("images/original")

# All .jpg and .jpeg file paths as strings
photo_paths = [str(file_path) for file_path in images_dir.glob("*") if file_path.suffix.lower() in [".jpg", ".jpeg"]]

def dms_to_decimal(dms, ref):
    degrees = float(dms.values[0].num) / float(dms.values[0].den)
    minutes = float(dms.values[1].num) / float(dms.values[1].den)
    seconds = float(dms.values[2].num) / float(dms.values[2].den)
    decimal = round(degrees + minutes / 60 + seconds / 3600, 2)
    if ref in ("S", "W"):
        decimal = -decimal
    return decimal

def format_dt(exif_dt):
    return datetime.strptime(str(exif_dt), "%Y:%m:%d %H:%M:%S")

def generate_photo_id()-> str:
    return uuid.uuid4().hex[:12]

def extract_lat_lon_time(photo_path):
    with open(photo_path, "rb") as f:
        tags = exifread.process_file(f, details=False)

    if (
        "GPS GPSLatitude" in tags
        and "GPS GPSLongitude" in tags
        and "EXIF DateTimeOriginal" in tags
    ):
        lat = dms_to_decimal(tags["GPS GPSLatitude"], str(tags["GPS GPSLatitudeRef"]))
        lon = dms_to_decimal(tags["GPS GPSLongitude"], str(tags["GPS GPSLongitudeRef"]))
        dt = format_dt(tags["EXIF DateTimeOriginal"])
        photo_id = generate_photo_id()
    else:
        lat, lon = 90.0, 0.0
        dt = datetime(2020, 1, 1, 23, 0, 0)
        photo_id = generate_photo_id()

    return {
        "id": photo_id,
        "photo_path": photo_path,
        "datetime_taken": dt,
        "lat": lat,
        "lon": lon,
    }

rows = [extract_lat_lon_time(path) for path in photo_paths]

photo_data_df = pd.DataFrame(rows, columns=["id", "photo_path", "datetime_taken", "lat", "lon"])

photo_data_df


,id,photo_path,datetime_taken,lat,lon
0,f955f64821f6,images\original\20260209_180728.jpg,2026-02-09 18:07:28,38.29,-85.68
1,aedb83a97f40,images\original\iceland_east_coastline.jpg,2017-09-26 15:55:11,64.45,-14.51
2,92c936d01569,images\original\iceland_west_coastline.jpg,2017-09-30 12:44:46,64.95,-23.42


In [21]:
photo_data_df["hour"] = photo_data_df['datetime_taken'].dt.round('h').dt.hour

photo_data_df.head()

,id,photo_path,datetime_taken,lat,lon,hour
0,f955f64821f6,images\original\20260209_180728.jpg,2026-02-09 18:07:28,38.29,-85.68,18
1,aedb83a97f40,images\original\iceland_east_coastline.jpg,2017-09-26 15:55:11,64.45,-14.51,16
2,92c936d01569,images\original\iceland_west_coastline.jpg,2017-09-30 12:44:46,64.95,-23.42,13


In [22]:
# add weather columns before adding weather data - fill with None

weather_cols = ["temp_f", "cloud_cover"]
for col in weather_cols:
    if col not in photo_data_df.columns:
        photo_data_df[col] = None
        
print(f"Loaded {len(photo_data_df):,} records")

Loaded 3 records


In [23]:
# add weather data to dataframe 

# Setup client 
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)


def get_weather(lat, lon, date_taken, hour=0):
    lat = round(float(lat), 2)
    lon = round(float(lon), 2)
    date_taken = pd.to_datetime(date_taken).strftime("%Y-%m-%d")
    hour = int(hour) if pd.notna(hour) else 0
    hour = hour if 0 <= hour <= 23 else 0

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date_taken,
        "end_date": date_taken,
        "hourly": ["temperature_2m", "cloud_cover"],
        "temperature_unit": "fahrenheit",
        "timezone": "auto",
    }

    try:
        responses = openmeteo.weather_api(
            "https://archive-api.open-meteo.com/v1/archive", params=params
        )
        hourly = responses[0].Hourly()
        temp = hourly.Variables(0).ValuesAsNumpy()
        cloud = hourly.Variables(1).ValuesAsNumpy()

        return {
            "temp_f": round(float(temp[hour]), 1) if len(temp) > hour else None,
            "cloud_cover": round(float(cloud[hour]), 1) if len(cloud) > hour else None,
        }
    except Exception as e:
        print(f"Weather API error ({lat}, {lon}, {date_taken} {hour}): {e}")
        return {"temp_f": None, "cloud_cover": None}



In [24]:
for idx, row in tqdm(photo_data_df.iterrows(), total=len(photo_data_df)):
    try:
        weather = get_weather(
            lat=row["lat"],
            lon=row["lon"],
            date_taken=row["datetime_taken"].strftime("%Y-%m-%d"),
            hour=row["hour"]
        )
        for key, value in weather.items():
            photo_data_df.at[idx, key] = value

    except Exception as e:
        print(f"Error at row {idx}: {e}")

photo_data_df


  0%|          | 0/3 [00:00<?, ?it/s]

,id,photo_path,datetime_taken,lat,lon,hour,temp_f,cloud_cover
0,f955f64821f6,images\original\20260209_180728.jpg,2026-02-09 18:07:28,38.29,-85.68,18,38.6,64.0
1,aedb83a97f40,images\original\iceland_east_coastline.jpg,2017-09-26 15:55:11,64.45,-14.51,16,52.2,100.0
2,92c936d01569,images\original\iceland_west_coastline.jpg,2017-09-30 12:44:46,64.95,-23.42,13,48.3,98.0


In [25]:
csv_filepath = Path("data/csv/photo_data.csv") 
csv_filepath.parent.mkdir(parents=True, exist_ok=True)  
photo_data_df.to_csv(csv_filepath) 

In [26]:
# create image from original image with photo exif data (from csv) displayed in lower left
    # create html page for image (jinja2 template?)
        # image (background)
        # data (white text or inverse of background?)
            # lat
            # lon
            # datetime
            # temp
        # other data: author name ? (in lower right)
    # save HTML image as a png or jpg in images/data_images


# Path to HTML Images Dir
html_images_dir = Path("html_images")
html_images_dir.mkdir(parents=True, exist_ok=True)

png_dir = Path("images/png_data_images")
png_dir.mkdir(parents=True, exist_ok=True)

full_photo_data_df = photo_data_df.copy()
full_photo_data_df["html_img_path"] = ""
full_photo_data_df["data_image_path"] = ""

for idx, row in full_photo_data_df.iterrows():
    photo_path = Path(row["photo_path"])
    photo_name = photo_path.stem

    # Missing EXIF was stored as lat=90, lon=0
    has_gps = not (row["lat"] == 90.0 and row["lon"] == 0.0)
    photo_date = row["datetime_taken"].strftime("%Y-%m-%d %H:%M")
    temp = row["temp_f"]
    temp_txt = f"{temp}°F" if pd.notna(temp) else "n/a"
    cloud = row["cloud_cover"]
    cloud_txt = f"{cloud}%" if pd.notna(cloud) else "n/a"

    html_content = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <style>
    body {{ margin: 0; background: #000; }}
    #image_div {{ position: relative; display: inline-block; }}
    img {{ display: block; max-width: 1600px; height: auto; }}
    #text_div {{
      position: absolute; bottom: 0; left: 0;
      padding: 48px 24px 20px 20px; color: #fff;
      font-family: sans-serif; font-size: 18px;
      background: linear-gradient(45deg, rgba(0,0,0,.75) 0%, rgba(0,0,0,0) 70%);
      text-shadow: 1px 1px 3px rgba(0,0,0,.6);
    }}
    #text_div p {{ margin: 4px 0; }}
  </style>
</head>
<body>
  <div id="image_div">
    <img src="{photo_path.resolve().as_uri()}" alt="{photo_name}">
    <div id="text_div">
      <p>Lat: {row["lat"] if has_gps else "n/a"}</p>
      <p>Lon: {row["lon"] if has_gps else "n/a"}</p>
      <p>{photo_date}</p>
      <p>Temp: {temp_txt}</p>
    </div>
  </div>
</body>
</html>"""

    html_filepath = html_images_dir / f"{photo_name}.html"
    html_filepath.write_text(html_content, encoding="utf-8")

    png_path = png_dir / f"{photo_name}_data_img.png"
    output_path = convert_html_to_png(html_filepath, png_path)

    full_photo_data_df.at[idx, "html_img_path"] = str(html_filepath)
    full_photo_data_df.at[idx, "data_image_path"] = str(png_path)
    print(f"Saved {output_path}")

full_photo_data_df.to_csv("data/csv/full_photo_data.csv", index=False)


# Later:
    # save data_image_path to column with that name in photo_data.csv 

    # photo maps using geopandas -> maps/

Saved images\png_data_images\20260209_180728_data_img.png
Saved images\png_data_images\iceland_east_coastline_data_img.png
Saved images\png_data_images\iceland_west_coastline_data_img.png


In [27]:
parts = []
for p in full_photo_data_df["data_image_path"]:
    if not p:
        continue
    rel = Path(p).as_posix()
    name = Path(p).stem
    parts.append(f'<div class="shot"><img src="{rel}" alt="{name}"></div>')

index_html = f"""<!DOCTYPE html>
<html>
<head>
  <meta charset="UTF-8">
  <title>Photo Data</title>
  <style>
    body {{ margin: 0; background: #111; }}
    .shot {{ margin: 24px auto; max-width: 1600px; }}
    img {{ width: 100%; height: auto; display: block; }}
  </style>
</head>
<body>
  {"".join(parts)}
</body>
</html>"""

Path("index.html").write_text(index_html, encoding="utf-8")

668